# Module 12 · Gene set enrichment

GSEApy prerank over the rankings module 11 exported.

Reads `gsea_prerank/gsea_config.json` — no model refitting, no dependency on
the R kernel. Everything about which contrasts exist and what they are called
comes from that file, so this notebook inherits `AXIS` without knowing about it.

| Section | |
|---|---|
| 01 | config and rankings |
| 02 | prerank |
| 03 | derive the reported blocks |
| 04 | Panel K |

---
## 01 · Config and rankings

**Why.** The ranking metric is the signed p-value, `-log10(p) * sign(logFC)`.
Ranking on evidence rather than magnitude keeps a lowly-expressed gene with a
large, noisy fold change out of the head of the list, where it would otherwise
drive the enrichment.

`RIBO_STRIP` is a config knob rather than a fixed choice — see section 02.

In [ ]:
import json, re, numpy as np, pandas as pd
from pathlib import Path

CFG = json.load(open(Path("gsea_prerank") / "gsea_config.json"))   # <- adjust if running elsewhere
AXIS      = CFG["AXIS"];  X_LAB = CFG["X_LAB"];  X_HEX = CFG["X_HEX"]
CONTRASTS = CFG["CONTRASTS"];  HEADERS = CFG["CONTRAST_HEADERS"]
DBS       = CFG["DBS"];  FDR_SIG = CFG["FDR_SIG"]
N_COMMON  = CFG["N_COMMON"];  N_UNIQUE = CFG["N_UNIQUE"]
RIBO_STRIP= CFG["RIBO_STRIP"]
OUT = Path(CFG["OUT"]);  FIG = Path(CFG["FIG"]);  FIG.mkdir(parents=True, exist_ok=True)

print(f"axis      : {AXIS}  ({X_LAB})")
print(f"contrasts : {CONTRASTS}")
print(f"databases : {DBS}   ribo-strip={RIBO_STRIP}   FDR<{FDR_SIG}")
print(f"out       : {OUT}")

In [ ]:
# ---- rankings: signed p-value from the limma tables -------------------------
RIBO = re.compile(r"^(RPL|RPS|MRPL|MRPS|RPLP\d|RPSA|FAU)", re.I)

def build_rnk(name):
    d = pd.read_csv(OUT.parent / "de" / f"{name}.csv")          # limma topTable export
    d = d.dropna(subset=["logFC", "P.Value"])
    d["metric"] = -np.log10(d["P.Value"].clip(lower=1e-300)) * np.sign(d["logFC"])
    if RIBO_STRIP:
        keep = ~d["gene"].astype(str).str.match(RIBO)
        print(f"  {name}: stripped {int((~keep).sum())} ribosomal genes")
        d = d[keep]
    r = d[["gene", "metric"]].sort_values("metric", ascending=False)
    r.to_csv(OUT / f"{name}.rnk", sep="\t", header=False, index=False)
    return r

rnks = {c: build_rnk(c) for c in CONTRASTS}
for c, r in rnks.items():
    print(f"  {c:22s} {len(r):>6} genes   range [{r.metric.min():.1f}, {r.metric.max():.1f}]")

---
## 02 · Prerank

**Why.** `RIBO_STRIP` is exposed deliberately. Ribosomal and translational
terms dominate many rankings and can crowd out everything else; stripping them
is defensible but it is a choice that changes the result, so it is a parameter
with both settings runnable rather than a silent filter.

**Database.** Reactome 2022.

In [ ]:
import gseapy as gp

def prerank(name, rnk):
    out = []
    for db in DBS:
        res = gp.prerank(rnk=rnk, gene_sets=db, outdir=None,
                         min_size=10, max_size=500, permutation_num=1000, seed=42)
        t = res.res2d.copy(); t["db"] = db; t["contrast"] = name
        out.append(t)
    t = pd.concat(out, ignore_index=True)
    suffix = "_noribo" if RIBO_STRIP else "_full"
    t.to_csv(OUT / f"gsea_{name}{suffix}.csv", index=False)
    return t

gsea = {c: prerank(c, rnks[c]) for c in CONTRASTS}
for c, t in gsea.items():
    print(f"  {c:22s} {len(t):>5} terms   sig: {int((pd.to_numeric(t['FDR q-val'],errors='coerce')<FDR_SIG).sum())}")

---
## 03 · Derive the reported blocks

**Why.** This is the rule the frozen `BLOCKS` literals in the source were
standing in for, written out so the selection is reproducible instead of
transcribed.

**Common block.** Significant in at least 3 contrasts, top N by mean NES.
**Unique block.** Significant in exactly 1 contrast, top N by |NES|.

Both N values are config, and the counts that were dropped are printed.

In [ ]:
# ---- assemble NES / FDR matrices -------------------------------------------
PREFIXES = tuple(f"{db}__" for db in DBS)
clean = lambda t: re.sub(r"\s*R-HSA-\d+\s*$", "", str(t).split("__", 1)[-1]).strip()

ARTIFACT = ["influenza","viral","virus","infection","infected","sars","covid","corona",
            "hiv","hcmv","measles","hepatitis","tuberculosis","leishman","disease"]

frames = {}
for c in CONTRASTS:
    d = gsea[c].copy()
    d["p"] = d["Term"].map(clean)
    d["NES"] = pd.to_numeric(d["NES"], errors="coerce")
    d["q"]   = pd.to_numeric(d["FDR q-val"], errors="coerce")
    d = d[~d["p"].str.lower().str.contains("|".join(ARTIFACT))]
    frames[c] = d.groupby("p").first()

paths = sorted(set().union(*[set(f.index) for f in frames.values()]))
nes = pd.DataFrame({c: frames[c].reindex(paths)["NES"] for c in CONTRASTS}, index=paths)
fdr = pd.DataFrame({c: frames[c].reindex(paths)["q"]   for c in CONTRASTS}, index=paths)

n_sig = (fdr < FDR_SIG).sum(axis=1)
print(f"  sig in >=3 (COMMON): {(n_sig>=3).sum()}   exactly 1 (UNIQUE): {(n_sig==1).sum()}")
print(f"  sig in 2: {(n_sig==2).sum()}   never: {(n_sig==0).sum()}")

In [ ]:
# ---- selection --------------------------------------------------------------
common = nes.loc[n_sig[n_sig >= 3].index].mean(axis=1)\
            .sort_values(ascending=False).head(N_COMMON).index.tolist()

unique = {}
for c in CONTRASTS:
    pool = n_sig[(n_sig == 1) & (fdr[c] < FDR_SIG)].index
    unique[c] = nes.loc[pool, c].abs().sort_values(ascending=False).head(N_UNIQUE).index.tolist()

# ---- display names: lookup with graceful fallback ---------------------------
# A new axis surfaces new pathways; anything absent here still renders, just longer.
DISPLAY_MAP = {
    "Major Pathway Of rRNA Processing In Nucleolus And Cytosol": "rRNA processing",
    "Nonsense Mediated Decay (NMD) Enhanced By Exon Junction Complex (EJC)": "NMD (EJC-enhanced)",
    "GTP Hydrolysis And Joining Of 60S Ribosomal Subunit": "60S subunit joining",
    "Cap-dependent Translation Initiation": "Cap-dependent translation",
    "Senescence-Associated Secretory Phenotype (SASP)": "SASP",
    "Gap-filling DNA Repair Synthesis And Ligation In TC-NER": "TC-NER gap-filling",
    "Formation Of TC-NER Pre-Incision Complex": "TC-NER pre-incision",
    "Global Genome Nucleotide Excision Repair (GG-NER)": "GG-NER",
    "E3 Ubiquitin Ligases Ubiquitinate Target Proteins": "E3 ubiquitin ligases",
    "Antiviral Mechanism By IFN-stimulated Genes": "IFN-stimulated genes",
    "Regulation Of Expression Of SLITs And ROBOs": "SLIT/ROBO expression",
}
def short(p, n=30):
    if p in DISPLAY_MAP: return DISPLAY_MAP[p]
    return p if len(p) <= n else p[:n-1] + "\u2026"

# ---- blocks: shared core first, then one block per contrast -----------------
BLOCK_COLORS = ["#5F5E5A", X_HEX, "#185FA5", "#993C1D", "#534AB7"]
BLOCKS = [("Shared", BLOCK_COLORS[0], [(p, short(p)) for p in common])]
for k, c in enumerate(CONTRASTS):
    if unique[c]:
        BLOCKS.append((HEADERS[k], BLOCK_COLORS[(k % 4) + 1],
                       [(p, short(p)) for p in unique[c]]))

BLOCKS_OVERRIDE = None      # <- set to a BLOCKS list to freeze a manuscript figure
if BLOCKS_OVERRIDE: BLOCKS = BLOCKS_OVERRIDE

for lab, col, ps in BLOCKS:
    print(f"  {lab:28s} ({len(ps)})")
    for full, sh in ps: print(f"      {sh}")

---
## 04 · Panel K

**Why.** Colour is NES **centered per pathway**, so the grid reads as
*which contrast is this pathway most enriched in* rather than *which pathway
has the biggest absolute NES*. Without centering, a handful of high-NES
pathways set the colour scale and the between-contrast differences the panel
exists to show are compressed into one shade.

In [ ]:
import matplotlib as mpl, matplotlib.pyplot as plt, matplotlib.cm as cm
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
mpl.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none",
    "font.family":"sans-serif","font.sans-serif":["Arial","Helvetica","DejaVu Sans"],
    "font.size":8,"axes.linewidth":0.5})

disp, NESl, FDRl, bidx = [], [], [], []
for bi,(_,_,ps) in enumerate(BLOCKS):
    for full, sh in ps:
        NESl.append([nes[c].get(full, np.nan) for c in CONTRASTS])
        FDRl.append([fdr[c].get(full, np.nan) for c in CONTRASTS])
        disp.append(sh); bidx.append(bi)
NES = np.array(NESl, float); FDR = np.array(FDRl, float)
NESc = NES - np.nanmean(NES, axis=1, keepdims=True)

vmax = max(np.nanmax(np.abs(NESc[~np.isnan(NESc)])), 0.3)
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cmap = mpl.colormaps["RdBu_r"]; nr, nc = NESc.shape

fig, ax = plt.subplots(figsize=(6.6, 0.40*nr + 1.9), dpi=300)
STRIP_W, STRIP_GAP = 0.55, 0.15
for i in range(nr):
    y = nr-1-i
    for j in range(nc):
        v = NESc[i, j]
        ax.add_patch(Rectangle((j-0.5, y-0.5), 1, 1,
            facecolor=cmap(norm(v)) if not np.isnan(v) else "#f5f5f5",
            edgecolor="white", lw=1.0, zorder=2))
        if not np.isnan(FDR[i, j]) and FDR[i, j] < FDR_SIG:
            ax.scatter(j, y, marker="o", s=70, facecolor="black",
                       edgecolor="white", linewidth=0.6, zorder=4)

sx = nc - 0.5 + STRIP_GAP
for bi,(label, color, _) in enumerate(BLOCKS):
    rows = [i for i,b in enumerate(bidx) if b == bi]
    if not rows: continue
    ytop = nr-1-min(rows)+0.5; ybot = nr-1-max(rows)-0.5
    ax.add_patch(Rectangle((-0.5, ybot), nc, ytop-ybot, fill=False,
                           edgecolor=color, lw=1.6, zorder=5))
    ax.add_patch(Rectangle((sx, ybot), STRIP_W, ytop-ybot, facecolor=color,
                           alpha=0.16, edgecolor="none", zorder=1, clip_on=False))
    ax.add_patch(Rectangle((sx, ybot), 0.06, ytop-ybot, facecolor=color,
                           edgecolor="none", zorder=2, clip_on=False))
    ax.text(sx+STRIP_W/2, (ytop+ybot)/2, label, rotation=270, ha="center",
            va="center", fontsize=7.5, color=color, zorder=6, clip_on=False)

ax.set_xlim(-0.5, sx+STRIP_W+0.1); ax.set_ylim(-0.5, nr-0.5)
ax.set_xticks(range(nc)); ax.set_xticklabels(HEADERS[:nc], fontsize=7.5,
    rotation=45, ha="right", rotation_mode="anchor")
ax.set_yticks(range(nr)); ax.set_yticklabels(disp[::-1], fontsize=8)
ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_aspect("equal"); ax.xaxis.set_ticks_position("bottom")

sm = cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.22, shrink=0.5)
cb.set_label("NES (centered per pathway)", fontsize=7.5); cb.ax.tick_params(labelsize=7)
ax.legend(handles=[Line2D([],[],marker="o",color="w",markerfacecolor="black",
    markeredgecolor="white",markersize=8,label=f"FDR < {FDR_SIG}")],
    loc="lower left", bbox_to_anchor=(1.10,-0.05), fontsize=7, frameon=False)

fig.tight_layout()
TAG = f"rowcentered_boxed_K_{AXIS}"
for ext in ("pdf","png","svg"):
    fig.savefig(FIG / f"gsea_grid_{TAG}.{ext}", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\u2713 saved gsea_grid_{TAG}.{{pdf,png,svg}}  ({nr} rows x {nc})   axis={AXIS}")
print("\nraw / centered NES per pathway (for caption):")
for i, nm in enumerate(disp):
    print(f"  {nm:28s} raw {np.array2string(NES[i],precision=2,floatmode='fixed')}"
          f"  ctr {np.array2string(NESc[i],precision=2,floatmode='fixed')}")